# 04 — Phi-4 Multimodal: Prompt Development for Nepali ASR Benchmarking

**Model:** `microsoft/Phi-4-multimodal-instruct`  
**Architecture:** Multimodal (text + image + audio), max 40s audio  
**Class:** `AutoModelForCausalLM` + `AutoProcessor` (trust_remote_code)  
**Audio input:** Uses `<|audio_1|>` placeholder token


## 0. Install Dependencies


In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg])

pip("transformers==4.48.2") # MUST be 4.48.2 for Phi-4 remote code!
pip("accelerate>=1.6")

pip("peft")          # Required by Phi-4 multimodal
pip("torchao>=0.16.0") # Required by latest PEFT
pip("soundfile>=0.12")
pip("scipy")
pip("librosa>=0.10")
pip("bitsandbytes>=0.45")
pip("sentencepiece")
pip("protobuf")
pip("backoff")  # Required by Phi-4 remote code



pip("jiwer>=3.1")
pip("jsonlines")
pip("pandas")
pip("tqdm")

print("\n✅ All dependencies installed.")


## A. Experiment Configuration


In [ ]:
import os, json, torch, gc
from datetime import datetime

MODEL_ID   = "microsoft/Phi-4-multimodal-instruct"
MODEL_REV  = "main"
QUANT      = "auto"

experiment_config = {
    "model_id":       MODEL_ID,
    "model_revision": MODEL_REV,
    "quantization":   QUANT,
    "random_seed":    42,
    "audio_sr":       16000,
    "batch_size":     1,
    "max_new_tokens": 256,
    "temperature":    0.0,
    "do_sample":      False,
    "timestamp":      datetime.now().isoformat(),
}

AUDIO_BASE = "/kaggle/input/datasets/panditaadarsh/llm-bechmarking-audio"
RESULTS_DIR = "/kaggle/working/results/prompt_dev/phi4_multimodal"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(f"{RESULTS_DIR}/run_config.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

print("Config saved →", RESULTS_DIR)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## B. Model Loading


In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor

print(f"Loading {MODEL_ID} ...")

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    _attn_implementation="eager",
)

# Phi-4 Multimodal uses a mixture of LoRAs. We MUST load the speech adapter!
print("Loading speech adapter...")
from huggingface_hub import snapshot_download
import os
adapter_path = snapshot_download(MODEL_ID, allow_patterns=["speech-lora/*"])
adapter_path = os.path.join(adapter_path, "speech-lora")
model.load_adapter(adapter_path, adapter_name="speech")
model.set_adapter("speech")

print("✅ Model loaded successfully.")


## B.1 — Sanity Check


In [ ]:
import soundfile as sf
import glob

audio_files = sorted(glob.glob(f"{AUDIO_BASE}/clean_nepali_200_flat/*.wav"))
if not audio_files:
    audio_files = sorted(glob.glob(f"{AUDIO_BASE}/clean_nepali_200_flat/*.mp3"))
test_audio_path = audio_files[0]
print(f"Testing: {test_audio_path}")

# Load audio
audio_data, samplerate = sf.read(test_audio_path)
print(f"Audio: {len(audio_data)/samplerate:.2f}s at {samplerate} Hz")

# Phi-4 strictly requires the full system + user + assistant structure
prompt = "<|system|>You are a helpful assistant.<|end|><|user|><|audio_1|>Transcribe the following Nepali audio into Nepali text.<|end|><|assistant|>"

    inputs = processor(
        text=prompt,
        audios=[(audio_data, samplerate)],
        return_tensors="pt"
    ).to(model.device)

from transformers import GenerationConfig
generation_config = GenerationConfig.from_pretrained(MODEL_ID)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        generation_config=generation_config,
    )

result = processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
print(f"\n📝 Model output:\n{result}")


## C. Prompt Templates

Note: Phi-4 uses `<|system|>...<|end|><|user|><|audio_1|>...<|end|><|assistant|>` as the chat format.


In [ ]:
SYS = "<|system|>You are a helpful assistant.<|end|>"
PROMPTS = {
    "L0_a": f"{SYS}<|user|><|audio_1|>Transcribe the following Nepali audio into Nepali text.<|end|><|assistant|>",
    "L1_a": (
        f"{SYS}<|user|><|audio_1|>You are a speech transcription system. "
        "Transcribe the following Nepali audio into Nepali text using Devanagari script. "
        "Produce a verbatim transcription. Do not translate. "
        "Return only the transcription, nothing else.<|end|><|assistant|>"
    ),
    "L1_b": (
        f"{SYS}<|user|><|audio_1|>Task: verbatim Nepali speech transcription.\n"
        "Language: Nepali (Devanagari script).\n"
        "Instructions: transcribe exactly what is spoken. Do not translate. "
        "Output only the transcription.<|end|><|assistant|>"
    ),
    "L2_a": (
        f"{SYS}<|user|><|audio_1|>Transcribe the spoken Nepali audio verbatim in Devanagari script. "
        "Preserve any English words in Latin script. "
        "Maintain the order of Nepali–English code-switching as spoken. "
        "Do not translate between languages. "
        "Keep fillers, repetitions, corrections, and incomplete words. "
        "Do not correct grammar. Do not infer inaudible words. "
        "Do not add timestamps, speaker labels, explanations, or confidence scores. "
        "Return only the transcription.<|end|><|assistant|>"
    ),
    "L2_b": (
        f"{SYS}<|user|><|audio_1|>You are a verbatim transcription system for Nepali speech.\n"
        "Rules:\n"
        "1. Write Nepali in Devanagari.\n"
        "2. Write English words in Latin script.\n"
        "3. Preserve code-switching order.\n"
        "4. Do not translate.\n"
        "5. Keep fillers, repetitions, corrections, incomplete words.\n"
        "6. Do not correct grammar.\n"
        "7. Do not guess inaudible words.\n"
        "8. No timestamps, no speaker labels, no explanations.\n"
        "9. Output only the transcription.<|end|><|assistant|>"
    ),
}
print(f"Defined {len(PROMPTS)} prompt variants.")


## D. Build Manifest


In [ ]:
import pandas as pd

def build_manifest(audio_dir, condition, max_files=None):
    '''Scan audio directory and build a manifest DataFrame, loading references from CSV if available.'''
    import glob, os
    
    # Try to load metadata
    metadata_df = None
    for meta_name in ["metadata.csv", "noisy_metadata.csv"]:
        meta_path = os.path.join(audio_dir, meta_name)
        if os.path.exists(meta_path):
            metadata_df = pd.read_csv(meta_path)
            # Ensure we have a consistent identifier to join on
            if "file" in metadata_df.columns:
                metadata_df["utterance_id"] = metadata_df["file"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])
            break
            
    files = sorted(glob.glob(f"{audio_dir}/**/*.wav", recursive=True)) +             sorted(glob.glob(f"{audio_dir}/**/*.mp3", recursive=True))
            
    if max_files:
        files = files[:max_files]
        
    records = []
    for fp in files:
        uid = os.path.splitext(os.path.basename(fp))[0]
        
        # Look up reference
        ref_text = ""
        if metadata_df is not None and "utterance_id" in metadata_df.columns:
            match = metadata_df[metadata_df["utterance_id"] == uid]
            if not match.empty:
                # Use label_normalized if available, else reference
                if "label_normalized" in match.columns:
                    ref_text = str(match.iloc[0]["label_normalized"])
                elif "reference" in match.columns:
                    ref_text = str(match.iloc[0]["reference"])
                    
        records.append({
            "utterance_id": uid,
            "audio_path": fp,
            "speech_condition": condition,
            "reference_raw": ref_text,
        })
    return pd.DataFrame(records)


manifest = pd.concat([
    build_manifest(f"{AUDIO_BASE}/clean_nepali_200_flat", "clean", 5),
    build_manifest(f"{AUDIO_BASE}/noisy_nepali_200", "noisy", 5),
    build_manifest(f"{AUDIO_BASE}/codeswitched_nepali_200_flat", "codeswitched", 5),
], ignore_index=True)
print(f"Pilot manifest: {len(manifest)} utterances")


## E. Batch Inference Pipeline


In [ ]:
import time, jsonlines, soundfile as sf
from tqdm.auto import tqdm

def transcribe_one(audio_path, prompt_text):
    audio_data, samplerate = sf.read(audio_path)
    inputs = processor(text=prompt_text, audios=[(audio_data, samplerate)], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=experiment_config["max_new_tokens"], generation_config=generation_config)
    return processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def clean_output(raw):
    text = raw.strip()
    for prefix in ["Transcription:", "Output:", "```", "**"]:
        if text.startswith(prefix): text = text[len(prefix):]
    return text.strip("`*\n ")

def run_pipeline(manifest_df, prompts_dict, output_file):
    output_path = f"{RESULTS_DIR}/{output_file}"
    completed = set()
    if os.path.exists(output_path):
        with jsonlines.open(output_path) as reader:
            for obj in reader: completed.add((obj["utterance_id"], obj["prompt_id"]))
    
    pbar = tqdm(total=len(manifest_df)*len(prompts_dict), initial=len(completed), desc="Inference")
    for _, row in manifest_df.iterrows():
        for pid, ptxt in prompts_dict.items():
            if (row["utterance_id"], pid) in completed: pbar.update(1); continue
            rec = {"model_id": MODEL_ID, "utterance_id": row["utterance_id"], "prompt_id": pid,
                   "prompt_level": pid.split("_")[0], "audio_path": row["audio_path"],
                   "speech_condition": row["speech_condition"], "reference_raw": row.get("reference_raw",""),
                   "status": "success", "raw_output": "", "cleaned_prediction": "",
                   "inference_seconds": 0, "timestamp": datetime.now().isoformat()}
            try:
                t0 = time.time()
                raw = transcribe_one(row["audio_path"], ptxt)
                rec["inference_seconds"] = round(time.time()-t0, 2)
                rec["raw_output"] = raw; rec["cleaned_prediction"] = clean_output(raw)
                if not rec["cleaned_prediction"]: rec["status"] = "empty_output"
            except torch.cuda.OutOfMemoryError:
                rec["status"] = "out_of_memory"; gc.collect(); torch.cuda.empty_cache()
            except Exception as e:
                rec["status"] = "inference_error"; rec["raw_output"] = str(e)
            with jsonlines.open(output_path, mode="a") as w: w.write(rec)
            completed.add((row["utterance_id"], pid)); pbar.update(1)
    pbar.close()
    print(f"\n✅ Done. {len(completed)} results → {output_path}")


## F. Run Inference


In [ ]:
run_pipeline(manifest, PROMPTS, "raw_predictions.jsonl")


## G. Compute Metrics


In [ ]:
from jiwer import wer, cer

results = []
with jsonlines.open(f"{RESULTS_DIR}/raw_predictions.jsonl") as reader:
    for obj in reader:
        if obj["status"]=="success" and obj.get("reference_raw"):
            try: obj.update({"wer": round(wer(obj["reference_raw"], obj["cleaned_prediction"]),4), "cer": round(cer(obj["reference_raw"], obj["cleaned_prediction"]),4)})
            except: obj.update({"wer":1.0,"cer":1.0})
        results.append(obj)

df = pd.DataFrame(results)
df.to_csv(f"{RESULTS_DIR}/utterance_metrics.csv", index=False)
if "wer" in df.columns:
    summary = df[df["status"]=="success"].groupby("prompt_id").agg(avg_wer=("wer","mean"),avg_cer=("cer","mean"),count=("utterance_id","count")).reset_index().sort_values("avg_wer")
    summary.to_csv(f"{RESULTS_DIR}/prompt_summary.csv", index=False)
    display(summary)
else:
    print(df["status"].value_counts())
